In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path.cwd() / "data" / "raw" / "acs"

acs_files = sorted(data_dir.glob("acs5_county_20??.csv"))

print("ACS CSV files found:")
for file in acs_files:
    print(file.name)

print("\nNumber of files:", len(acs_files))

In [ ]:
acs_sample = pd.read_csv(
    acs_files[0],
    dtype=str
)

print("Dataset shape:")
print(acs_sample.shape)

print("\nColumn names:")
print(acs_sample.columns.tolist())

print("\nFirst five rows:")
print(acs_sample.head().to_string(index=False))

print("\nMissing values by column:")
print(acs_sample.isna().sum())

In [ ]:
acs_variables = [
    "B19013_001E",
    "B25077_001E",
    "B25064_001E"
]

for col in acs_variables:
    numeric_values = pd.to_numeric(
        acs_sample[col],
        errors="coerce"
    )

    print(f"\nVariable: {col}")

    print("Non-numeric values:",
          numeric_values.isna().sum())

    print("Negative values:",
          (numeric_values < 0).sum())

    print("Minimum value:",
          numeric_values.min())

    print("Maximum value:",
          numeric_values.max())

    print("Most common negative values:")
    print(
        numeric_values[numeric_values < 0]
        .value_counts()
        .head(10)
    )

In [ ]:
acs_sample["county_fips"] = (
    acs_sample["state"].str.zfill(2)
    + acs_sample["county"].str.zfill(3)
)

print("Sample county identifiers:")
print(
    acs_sample[
        ["NAME", "state", "county", "county_fips"]
    ].head(10)
)

print("\nCounty FIPS lengths:")
print(
    acs_sample["county_fips"]
    .str.len()
    .value_counts()
)

print("\nDuplicate county identifiers:")
print(
    acs_sample["county_fips"].duplicated().sum()
)

In [ ]:
acs_dfs = []

for file in acs_files:

    year = int(file.stem.split("_")[-1])

    df = pd.read_csv(file, dtype=str)

    df["county_fips"] = (
        df["state"].str.zfill(2)
        + df["county"].str.zfill(3)
    )

    df = df.rename(columns={
        "NAME": "county_name",
        "B19013_001E": "median_household_income",
        "B25077_001E": "median_home_value",
        "B25064_001E": "median_gross_rent"
    })

    value_columns = [
        "median_household_income",
        "median_home_value",
        "median_gross_rent"
    ]

    for col in value_columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df.loc[df[col] < 0, col] = float("nan")

    df["year"] = year

    df = df[
        [
            "county_fips",
            "county_name",
            "year",
            "median_household_income",
            "median_home_value",
            "median_gross_rent"
        ]
    ]

    acs_dfs.append(df)

acs_combined = pd.concat(
    acs_dfs,
    ignore_index=True
)

acs_combined = acs_combined.sort_values(
    ["county_fips", "year"]
).reset_index(drop=True)

print("Combined dataset shape:", acs_combined.shape)

print("\nFirst 10 records:")
print(acs_combined.head(10))

print("\nRecords by year:")
print(
    acs_combined["year"]
    .value_counts()
    .sort_index()
)

In [ ]:
print("Missing values by column:")
print(acs_combined.isna().sum())

print("\nDuplicate county-year records:")
print(
    acs_combined.duplicated(
        ["county_fips", "year"]
    ).sum()
)

print("\nCounty FIPS lengths:")
print(
    acs_combined["county_fips"]
    .str.len()
    .value_counts()
)

print("\nSummary statistics:")
print(
    acs_combined[
        [
            "median_household_income",
            "median_home_value",
            "median_gross_rent"
        ]
    ].describe()
)

print("\nMissing values by year:")
print(
    acs_combined.groupby("year")[
        [
            "median_household_income",
            "median_home_value",
            "median_gross_rent"
        ]
    ].apply(lambda x: x.isna().sum())
)

In [ ]:
acs_clean = acs_combined.copy()

acs_clean["home_value_income_ratio"] = (
    acs_clean["median_home_value"]
    / acs_clean["median_household_income"]
)

print("Final dataset shape:")
print(acs_clean.shape)

print("\nFirst 10 records:")
print(acs_clean.head(10))

print("\nMissing affordability ratios:")
print(
    acs_clean["home_value_income_ratio"]
    .isna().sum()
)

print("\nAffordability ratio summary:")
print(
    acs_clean["home_value_income_ratio"]
    .describe()
)

In [ ]:
from pathlib import Path

processed_dir = Path.cwd() / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "acs_county_housing_income_2019_2024.csv"

acs_clean.to_csv(
    output_file,
    index=False
)

print("Saved cleaned Census dataset to:")
print(output_file)